# US sector ETF cross-sectional momentum

Long the top 3 sectors and short the bottom 3 sectors by 12-1 month momentum. Signals are always computed from the original Select Sector SPDR ETFs in `configs/us_sector_etf.csv`.

Trade expression is configurable:

- `original_short`: long winners and short losers directly using the original sector ETFs.
- `leveraged_etf`: long winners through long leveraged ETFs and express bearish views through inverse ETFs. With `PRESERVE_EXPOSURE = True`, a 2x ETF receives half the target weight and a 3x ETF receives one third.

The sector baseline is the 11 Select Sector SPDR suite. Leveraged and inverse mappings use currently listed broad sector products where available; blanks mean no broad ETF is mapped for that leverage bucket.

In [ ]:
import pandas as pd

from alpha_lab.backtest.metrics import summary
from alpha_lab.backtest.sector_momentum import (
    express_sector_views,
    sector_momentum_signal,
    top_bottom_view_weights,
)
from alpha_lab.backtest.vector import run_backtest
from alpha_lab.data.loaders.yfinance import load_prices
from alpha_lab.reporting.charts import drawdown_chart, equity_curve, heatmap_monthly
from alpha_lab.utils.paths import CONFIGS_DIR

In [ ]:
START = "2010-01-01"
END = None

LOOKBACK_MONTHS = 12
SKIP_MONTHS = 1
TOP_N = 3
BOTTOM_N = 3

# Use "ME" for monthly rebalance or "W-FRI" for weekly rebalance.
REBALANCE = "ME"

# Use "original_short" or "leveraged_etf".
EXPRESSION_MODE = "original_short"
EXPRESSION_LEVERAGE = 2
PRESERVE_EXPOSURE = True

COSTS_BPS = 1.0
SLIPPAGE_BPS = 2.0

In [ ]:
universe = pd.read_csv(CONFIGS_DIR / "us_sector_etf.csv").fillna("")
universe

In [ ]:
if EXPRESSION_MODE == "leveraged_etf":
    long_col = f"long_{EXPRESSION_LEVERAGE}x_etf"
    inverse_col = f"inverse_{EXPRESSION_LEVERAGE}x_etf"
    active_universe = universe[(universe[long_col] != "") & (universe[inverse_col] != "")].copy()
else:
    active_universe = universe.copy()

active_universe[["sector", "signal_etf"]]

In [ ]:
signal_tickers = active_universe["signal_etf"].tolist()
signal_prices = load_prices(signal_tickers, start=START, end=END)

momentum = sector_momentum_signal(
    signal_prices,
    lookback_months=LOOKBACK_MONTHS,
    skip_months=SKIP_MONTHS,
)
view_weights = top_bottom_view_weights(momentum, top_n=TOP_N, bottom_n=BOTTOM_N)
view_weights.tail()

In [ ]:
trade_weights = express_sector_views(
    view_weights,
    active_universe,
    mode=EXPRESSION_MODE,
    leverage=EXPRESSION_LEVERAGE,
    preserve_exposure=PRESERVE_EXPOSURE,
)
trade_tickers = trade_weights.columns.tolist()
trade_prices = load_prices(trade_tickers, start=START, end=END)
trade_weights.tail()

In [ ]:
res = run_backtest(
    trade_weights,
    trade_prices,
    rebalance=REBALANCE,
    costs_bps=COSTS_BPS,
    slippage_bps=SLIPPAGE_BPS,
)
pd.Series(summary(res.returns))

In [ ]:
equity_curve(res.returns, name=f"sector momentum {EXPRESSION_MODE}")

In [ ]:
drawdown_chart(res.returns)

In [ ]:
heatmap_monthly(res.returns)

In [ ]:
latest_views = pd.concat(
    [momentum.iloc[-1].rename("momentum"), view_weights.iloc[-1].rename("view_weight")],
    axis=1,
).sort_values("momentum", ascending=False)
latest_views